# 03 · Выравнивание на предпочтениях

**Цель:** та же, что в `01-sft`, но через пары «эталон / плохой ответ» на одну и ту же ситуацию. Плохой ответ показывает границу явно: вот формулировка гипотезы за студента, вот переписанное введение, вот выдуманные источники — именно то, что автопроверки ловят плохо, а судья хорошо.

Промпт пары — ситуация вместе с обменом `select_skill`, как в проде; выбранный и отвергнутый — конечная реплика. `START_FROM` позволяет учить поверх SFT-адаптера (связка SFT → DPO), иначе — с базовой модели. `METHOD` переключает ORPO, DPO, SimPO, KTO. Математика — `books/03-alignment.pdf`.

In [ ]:
from common import (MODEL_ID, SYSTEM, TOOLS, RUNS, SHOWCASE, load_rows, user_message, pairs_for, evaluate, judge,
                    judge_rate, fmt, table, show_case)

import json
import torch
from datasets import Dataset
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import LoraConfig, PeftModel
from vlmkit import Sample, memory_report, evaluate as ev
from vlmkit.compat import alignment_trainer, available_alignment, supported

print("доступно в вашем trl:", available_alignment())

METHOD = "orpo"                 # или simpo / dpo / kto
START_FROM = RUNS / "sft"       # None — с базовой модели; путь — поверх SFT-адаптера
RUN_NAME = f"sft-{METHOD}" if START_FROM else METHOD

golden = load_rows("golden")
train = load_rows("train")

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

if START_FROM is not None and START_FROM.exists():
    model = PeftModel.from_pretrained(model, str(START_FROM), is_trainable=True)
    print("стартуем с адаптера", START_FROM.name)
else:
    START_FROM = None
    print("стартуем с базовой модели")

## До

Помимо метрик на голд-сете — preference accuracy: доля отложенных пар, где модель ставит эталону большую вероятность на токен, чем плохому ответу. Это то, что trl логирует в обучении как `rewards/accuracies`, только на отложенных ситуациях и без опорной модели.

In [ ]:
def report(results, per_row, verdicts=None, ids=SHOWCASE):
    """Полные ответы модели на показательные ситуации с проверками и вердиктом судьи."""
    for i, row in enumerate(golden):
        if row["id"] in ids:
            show_case(row, results[i], per_row[i]["checks"], verdicts[i] if verdicts else None)

before, before_rows, before_summary = evaluate(model, processor, golden)
before_verdicts = judge(model, processor, golden, [r["text"] for r in before])
before_summary["judge_pass"] = judge_rate(before_verdicts)

good = [Sample.from_qa(user_message(r), r["answer"]) for r in golden]
bad = [Sample.from_qa(user_message(r), r["rejected"]) for r in golden]
pref_before = ev.preference_accuracy(model, processor, good, bad, system=SYSTEM)
print(fmt(before_summary), f"судья {before_summary['judge_pass']:.0%}")
print("preference accuracy на голд-сете:", pref_before)
report(before, before_rows, before_verdicts)

## Пары

Диалоговый формат: trl сам прогоняет пары через chat template модели, и обучение видит тот же текст, что инференс. Если ваша версия trl не принимает реплику `tool` внутри промпта, поставьте `with_skill=False` — промпт станет «system + запрос», без обмена с навыком.

In [ ]:
pairs = Dataset.from_list(pairs_for(train, with_skill=True))
print(pairs)
p = pairs[0]
print("\nроли промпта:", [m["role"] for m in p["prompt"]])
print(f"ВЫБРАННЫЙ:   {p['chosen'][0]['content'][:200]}…")
print(f"ОТВЕРГНУТЫЙ: {p['rejected'][0]['content'][:200]}…")

## Тренер

`alignment_trainer` находит класс под метод в вашей версии trl (в trl 1.x ORPO и CPO лежат в `trl.experimental`, SimPO — это CPO с `loss_type="simpo"`, `cpo_alpha=0`). Тренеру передаётся токенизатор: пары текстовые, а `ORPOTrainer` и `CPOTrainer` берут `pad_token_id` напрямую. Поверх SFT-адаптера `peft_config` не передаётся — тренер учит уже загруженный адаптер.

`beta`: у DPO — сила KL-штрафа (0.1), у ORPO — вес слагаемого с отношением шансов (0.1), у SimPO — масштаб награды (2.0).

In [ ]:
Config, TrainerCls, extra = alignment_trainer(METHOD)
print(f"{METHOD} → {TrainerCls.__module__}.{TrainerCls.__name__} {extra}")

lora = None if START_FROM else LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True, bias="none", task_type="CAUSAL_LM",
)

if METHOD == "kto":      # не пары, а отдельные ответы с меткой
    train_data = Dataset.from_list(
        [{"prompt": r["prompt"], "completion": r["chosen"],   "label": True}  for r in pairs]
        + [{"prompt": r["prompt"], "completion": r["rejected"], "label": False} for r in pairs]
    )
    processor.tokenizer.padding_side = "left"
else:
    train_data = pairs

args = Config(**supported(Config, {
    **extra,
    "output_dir": str(RUNS / RUN_NAME),
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "num_train_epochs": 2,
    "learning_rate": 5e-6,
    "beta": 2.0 if METHOD == "simpo" else 0.1,
    "bf16": True,
    "max_length": 4096,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "logging_steps": 5,
    "save_strategy": "no",
    "report_to": [],
    "remove_unused_columns": False,
}))

trainer = TrainerCls(
    model=model, args=args, train_dataset=train_data,
    processing_class=processor.tokenizer,
    peft_config=lora,
)
trainer.train()
model = trainer.model

## После

In [ ]:
model.eval()
after, after_rows, after_summary = evaluate(model, processor, golden)
after_verdicts = judge(model, processor, golden, [r["text"] for r in after])
after_summary["judge_pass"] = judge_rate(after_verdicts)
report(after, after_rows, after_verdicts)

pref_after = ev.preference_accuracy(model, processor, good, bad, system=SYSTEM)
print(f"\npreference accuracy: {pref_before['accuracy']:.0%} → {pref_after['accuracy']:.0%}, "
      f"зазор на токен {pref_before['margin']:+.3f} → {pref_after['margin']:+.3f}")
table({"до": before_summary, RUN_NAME: after_summary})

model.save_pretrained(str(RUNS / RUN_NAME))
(RUNS / f"{RUN_NAME}.json").write_text(json.dumps(after_summary, ensure_ascii=False, indent=2), encoding="utf-8")

## Как читать

Выравнивание правит сложившееся поведение, а не создаёт новое: с базовой модели на 257 парах эффект будет скромным, поверх SFT — заметным, особенно по judge_pass. Если preference accuracy выросла, а автопроверки нет — модель научилась предпочитать эталон, но в генерации до него не доходит; помогает вторая эпоха или `beta` поменьше у ORPO.